# 01 - Dataset audit

Objective: inspect Parquet file metadata and verify the source inventory before analysis. The audit report at `reports/dataset_audit_report.md` contains the full row-level quality findings.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))


In [ ]:
import pandas as pd
import pyarrow.parquet as pq

raw = ROOT / "data" / "raw" / "india" / "stocks"
files = sorted(raw.glob("nse_*.parquet"))
rows = []
for path in files:
    parquet = pq.ParquetFile(path)
    rows.append({"file": path.name, "bytes": path.stat().st_size, "rows": parquet.metadata.num_rows, "columns": parquet.metadata.num_columns, "schema": str(parquet.schema_arrow)})
summary = pd.DataFrame(rows)
display(summary[["file", "bytes", "rows", "columns"]])
display(summary.loc[0, "schema"])
shard = pq.ParquetFile(raw / "train-00000.parquet")
print("Separate shard:", shard.metadata.num_rows, "rows")
display(shard.schema_arrow)

Interpretation: filenames alone do not prove source or field semantics. Consult the audit report for the shard?s mixed timestamp cadence and price discrepancies. Do not mix it with annual NSE records without resolving provenance and price basis.

Conclusion: annual files are suitable for the India daily pipeline with the documented caveats; US, FX, and macro data are missing.